# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://mlcroissant.io/) library, based on the Croissant metadata standard.

### Dataset Source
FAIR² Dataset accessible via Croissant schema:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level metadata
print(f"Dataset Title: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets (`cr:RecordSet`), fields, and their `@id` references. This helps identify the structure and schema before extraction.

Use `.record_sets` and `.fields` attributes in Croissant metadata. All IDs are referenced by their `@id`.

In [ ]:
# List all record sets in the dataset
print("Available Record Sets and their Fields:")
for rs in metadata.record_sets:
    print(f"\nRecord Set Name: {rs.name}")
    print(f"@id: {rs.id}")
    if hasattr(rs, 'description'):
        print(f"Description: {rs.description}")
    # List all fields of the record set
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'N/A')})")

## 3. Data Extraction

Load data from specific record set(s) into pandas DataFrames for analysis. All Croissant entities are referenced by their `@id`. One or more record sets may be available in the FAIR² dataset.

In [ ]:
# Compile all RecordSet `@id`s in the package
record_sets = [rs.id for rs in metadata.record_sets]
print("Record Set IDs:", record_sets)

# Load records into pandas DataFrames, keyed by their @id
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Records loaded: {len(records)}. Columns: {dataframes[record_set_id].columns.tolist()}")

# Display the first DataFrame's columns and preview data
if record_sets:
    main_rs_id = record_sets[0]
    print(f"\nColumns in {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Common data processing steps, such as filtering on record values, normalization, and aggregation are performed here.

We'll select a numeric field by `@id` from the record set, then demonstrate filtering and normalization using pandas.

In [ ]:
# --- Choose a record set and a numeric field ---
# We'll pick the main record set and attempt to find a numeric field
main_rs = None
for rs in metadata.record_sets:
    # We'll try to use the first record set that contains at least one Integer/Float field
    numeric_candidate = None
    for field in rs.fields:
        dt = getattr(field, 'data_type', None)
        if dt in ['schema:Integer', 'schema:Float', 'schema:Number']:
            numeric_candidate = field
            break
    if numeric_candidate is not None:
        main_rs = rs
        break

if main_rs is None:
    raise Exception("No numerical field found in any record set.")

# Use the field's @id
numeric_field_id = numeric_candidate.id
group_field = None
# Attempt to select another non-numeric field (often a category)
for f in main_rs.fields:
    if f.id != numeric_field_id and getattr(f, 'data_type', '') == 'schema:Text':
        group_field = f.id
        break

# Use the DataFrame associated with main_rs.id
df = dataframes[main_rs.id]

print(f"Selected numeric field (@id): {numeric_field_id}")
if group_field:
    print(f"Selected group field (@id): {group_field}")
else:
    print("No group (categorical) field detected.")

# Filter records: take those with the numeric value > threshold (using threshold=10 as a general example)
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\n{numeric_field_id} normalized (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: group by group_field and show aggregated mean
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        grouped_df.columns = [group_field, f"mean_{numeric_field_id}"]
        print(f"\nGrouped by {group_field} (mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization

Visualize distributions or relationships between selected fields using matplotlib or seaborn. We'll demonstrate a histogram for the numeric field and, if a group/categorical field is present, a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot if group exists
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore a Croissant-compliant dataset using the `mlcroissant` Python library. By referencing all schema entities (record sets, fields, etc.) via their `@id`, you can reliably explore, preprocess, and visualize the dataset structure and content. Adjust the EDA and visualizations as needed for research questions or to target other record sets or variables.